In [0]:
%run "../config/schema_configs"

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# --- pipeline 1: CLICKSTREAM  ---
print("\n🔄 Iniciando Stream para Clickstream...")

# configura o caminho da camada Bronze
cfg_clickstream = BRONZE_CONFIG["clickstream"]

# FIX PARA SERVERLESS: Forçar a deleção profunda das pastas de metadados do Auto Loader
try:
    print("🧹 Removendo rastros antigos de inferência...")
    dbutils.fs.rm(cfg_clickstream["schema_location"], recurse=True)
    dbutils.fs.rm(cfg_clickstream["checkpoint"], recurse=True)
    spark.sql(f"DROP TABLE IF EXISTS {cfg_clickstream['target_table']}")
    print(" ✅ Pastas e tabelas limpas.")
except Exception as e:
    print(f"Aviso na limpeza: {e}")

# Leitura com Auto Loader
df_stream_clickstream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", cfg_clickstream["schema_location"]) \
    .option("cloudFiles.inferColumnTypes", "true") \
    .load(cfg_clickstream["source_path"])

# Injeção de metadados de linhagem
df_bronze_clickstream = df_stream_clickstream \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_datasource", lit(cfg_clickstream["source_path"]))

# Gravação incremental na tabela Bronze
query_clickstream = df_bronze_clickstream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", cfg_clickstream["checkpoint"]) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable(cfg_clickstream["target_table"])

query_clickstream.awaitTermination()
print(f" ✅ Clickstream ingerido com sucesso na tabela: {cfg_clickstream['target_table']}")

# --- pipeline 2: SALES  ---
print("\n🔄 Iniciando Stream para Sales...")

# configura o caminho da camada Bronze
cfg_sales = BRONZE_CONFIG["sales"]

# Leitura com Auto Loader
df_stream_sales = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", cfg_sales["schema_location"]) \
    .load(cfg_sales["source_path"])

# Injeção de metadados de linhagem
df_bronze_sales = df_stream_sales \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_datasource", lit(cfg_sales["source_path"]))    

query_sales = df_bronze_sales.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", cfg_sales["checkpoint"]) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable(cfg_sales["target_table"]) 

query_sales.awaitTermination()
print(f" ✅ Sales ingerido com sucesso na tabela: {cfg_sales['target_table']}")  

print("\n✨ Camada Bronze concluída com sucesso!")